# 00 · Connection

Block 0 check: connect to MCP, find the tools we need, read the stage, acquire one HAADF image and read it back from Tiled.

**Before running (twin):** `uv run python scripts/twin.py start` in a terminal. Stop it later with `uv run python scripts/twin.py stop`.

**Real microscope:** set `STEM_REAL_MCP_URL` in `.env` and `TARGET = "real"`.

In [ ]:
TARGET = "twin"   # "twin" or "real"

In [ ]:
import json

import matplotlib.pyplot as plt

from stem_agent.config import TOOLS
from stem_agent.data import tiled_io
from stem_agent.mcp import session

client = session.connect(TARGET)
tools = await client.list_tools()
print(f"{client.url}: {len(tools)} tools")

## Tools we need
Every short name in `config.TOOLS` must map to a tool on the server.

In [ ]:
missing = []
for short in TOOLS:
    name = session.tool_name(short)
    tool = tools.get(name)
    args = list((tool.inputSchema or {}).get("properties", {})) if tool else []
    print(f"{'ok     ' if tool else 'MISSING'} {short:<20} {name}({', '.join(args)})")
    if tool is None:
        missing.append(short)
print("missing:", missing or "none")

## Read-only calls

In [ ]:
stage = await session.call("get_stage")
fov = await session.call("get_fov")
print("stage [x, y, z, alpha, beta]:", stage)
print(f"FOV: {fov * 1e9:.1f} nm")

## One HAADF image
MCP returns the Tiled key; the full image is read from Tiled over HTTP.

In [ ]:
uri = json.loads(await session.call("get_tiled_config"))["uri"]
tiled_io.connect(uri)

key = await session.call("acquire_image", ["haadf"])
image = tiled_io.read_image(key)
print(f"Tiled {uri}\nkey {key}\nshape {image.shape}, range [{image.min():.3g}, {image.max():.3g}]")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(image, cmap="gray")
ax.set_title(f"HAADF ({TARGET})")
ax.axis("off")
plt.show()